<a href="https://colab.research.google.com/github/jacobwechuli/trustworthy_chat/blob/main/trustworthyChat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit llama-index llama-index-llms-cleanlab llama-index-llms-huggingface llama-parse gradio

In [ ]:
"""
Helper utilities for wiring Cleanlab's TLM trustworthiness score
into a LlamaIndex query engine via LlamaIndex's instrumentation API.

Reference:
https://developers.llamaindex.ai/python/examples/cookbooks/cleanlab_tlm_rag/
"""

from typing import ClassVar, Dict, List

from llama_index.core.instrumentation import get_dispatcher
from llama_index.core.instrumentation.event_handlers import BaseEventHandler
from llama_index.core.instrumentation.events import BaseEvent
from llama_index.core.instrumentation.events.llm import LLMCompletionEndEvent


class GetTrustworthinessScore(BaseEventHandler):
    """Captures the trustworthiness_score (and explanation, if requested)
    that Cleanlab's TLM attaches to every LLM completion."""

    events: ClassVar[List[BaseEvent]] = []
    trustworthiness_score: float = 0.0
    explanation: str = ""

    @classmethod
    def class_name(cls) -> str:
        return "GetTrustworthinessScore"

    def handle(self, event: BaseEvent) -> Dict:
        if isinstance(event, LLMCompletionEndEvent):
            kwargs = event.response.additional_kwargs or {}
            self.trustworthiness_score = kwargs.get("trustworthiness_score", 0.0)
            self.explanation = kwargs.get("explanation", "")
            self.events.append(event)
        return {}


def setup_trustworthiness_handler() -> GetTrustworthinessScore:
    """Registers (once) a GetTrustworthinessScore handler on the root
    LlamaIndex dispatcher and returns it so callers can read
    `.trustworthiness_score` / `.explanation` after each query."""

    root_dispatcher = get_dispatcher()

    # Avoid stacking duplicate handlers if this is called more than once
    # (e.g. Streamlit re-runs the script on every interaction).
    for handler in root_dispatcher.event_handlers:
        if isinstance(handler, GetTrustworthinessScore):
            return handler

    event_handler = GetTrustworthinessScore()
    root_dispatcher.add_event_handler(event_handler)
    return event_handler


def outputs_with_trustworthiness(response, event_handler: GetTrustworthinessScore):
    """Given a query engine response and the registered event handler,
    return (answer_text, trustworthiness_score, explanation)."""

    response_str = response.response
    trustworthiness_score = event_handler.trustworthiness_score
    explanation = event_handler.explanation or "No explanation available."
    return response_str, trustworthiness_score, explanation

In [ ]:
import os
import uuid

import gradio as gr

from llama_index.core import Settings, PromptTemplate
from llama_index.core import VectorStoreIndex
from llama_index.llms.cleanlab import CleanlabTLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# from utils import setup_trustworthiness_handler, outputs_with_trustworthiness

MODEL_NAME = "gpt-4o"

QA_PROMPT_TMPL_STR = (
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information above I want you to think step by step to "
    "answer the query in a crisp manner, in case you don't know the answer "
    "say 'I don't know!'.\n"
    "Query: {query_str}\n"
    "Answer: "
)


def load_llm():
    api_key = os.environ.get("CLEANLAB_API_KEY")
    if not api_key:
        raise gr.Error(
            "CLEANLAB_API_KEY is not set. Add it to your environment before "
            "starting the app (get a free key at https://cleanlab.ai/)."
        )
    options = {"model": MODEL_NAME, "max_tokens": 256, "log": ["explanation"]}
    return CleanlabTLM(api_key=api_key, options=options)


def build_index(file_path):
    """Parse + index a single PDF, returning a ready-to-query query engine."""
    if not os.environ.get("LLAMA_CLOUD_API_KEY"):
        raise gr.Error(
            "LLAMA_CLOUD_API_KEY is not set. LlamaParse needs it to parse PDFs "
            "(get a free key at https://cloud.llamaindex.ai/)."
        )
    try:
        from llama_parse import LlamaParse

        parser = LlamaParse(result_type="markdown")
        docs = parser.load_data(file_path)
    except Exception as e:
        raise gr.Error(f"Error parsing PDF: {e}")

    Settings.embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-large-en-v1.5", trust_remote_code=True
    )
    Settings.llm = load_llm()

    index = VectorStoreIndex.from_documents(docs, show_progress=True)
    query_engine = index.as_query_engine()
    query_engine.update_prompts(
        {"response_synthesizer:text_qa_template": PromptTemplate(QA_PROMPT_TMPL_STR)}
    )
    return query_engine


# --------------------------------------------------------------------------
# Event handlers
# --------------------------------------------------------------------------
def handle_upload(file, file_cache, session_id):
    """Index the uploaded PDF (or reuse a cached index) and unlock chat input."""
    if file is None:
        return file_cache, "No file uploaded.", gr.update(interactive=False)

    file_key = f"{session_id}-{os.path.basename(file.name)}"

    if file_key not in file_cache:
        query_engine = build_index(file.name)
        file_cache[file_key] = query_engine

    file_cache["__active__"] = file_key
    status = f"✅ **{os.path.basename(file.name)}** indexed and ready to chat."
    return file_cache, status, gr.update(interactive=True, placeholder="Ask something about your document...")


def respond(message, chat_history, file_cache, event_handler):
    """Stream the answer word-by-word, then reveal the trustworthiness score."""
    active_key = file_cache.get("__active__")
    if not active_key or active_key not in file_cache:
        chat_history = chat_history + [(message, "Please upload and index a PDF first.")]
        yield "", chat_history, ""
        return

    query_engine = file_cache[active_key]
    response = query_engine.query(message)
    response_str, trustworthiness_score, explanation = outputs_with_trustworthiness(
        response, event_handler
    )

    # Simulated streaming: TLM scores the *completed* response, so the score
    # can only be shown once generation is done, but the text itself can be
    # revealed progressively for a nicer chat feel.
    chat_history = chat_history + [(message, "")]
    partial = ""
    for word in response_str.split():
        partial += word + " "
        chat_history[-1] = (message, partial)
        yield "", chat_history, ""

    chat_history[-1] = (message, response_str)
    score_html = f"""
    <div style="padding:10px 20px;border-radius:10px;background:rgba(0,180,180,0.15);
                display:inline-block;margin:6px 0;">
        <strong>Trustworthiness Score:</strong> {trustworthiness_score:.2f}
    </div>
    <p><strong>Reasoning:</strong> {explanation}</p>
    """
    yield "", chat_history, score_html


def clear_chat():
    return [], ""


# --------------------------------------------------------------------------
# UI layout
# --------------------------------------------------------------------------
with gr.Blocks(title="Chat with Docs (Cleanlab TLM)") as demo:
    session_id_state = gr.State(str(uuid.uuid4()))
    file_cache_state = gr.State({})
    # Registered once per app process (not per user session, mirroring the
    # single root LlamaIndex dispatcher utils.py wires this into).
    event_handler_state = gr.State(setup_trustworthiness_handler())

    gr.Markdown(f"## Chat with your Docs (powered by {MODEL_NAME} via Cleanlab TLM)")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Add your document")
            file_input = gr.File(label="Upload a PDF", file_types=[".pdf"])
            upload_status = gr.Markdown("Upload a PDF to get started.")
            clear_btn = gr.Button("Clear chat ↺")

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=450)
            score_display = gr.HTML()
            msg = gr.Textbox(
                placeholder="Upload a PDF first...",
                interactive=False,
                show_label=False,
            )

    file_input.upload(
        handle_upload,
        inputs=[file_input, file_cache_state, session_id_state],
        outputs=[file_cache_state, upload_status, msg],
    )

    msg.submit(
        respond,
        inputs=[msg, chatbot, file_cache_state, event_handler_state],
        outputs=[msg, chatbot, score_display],
    )

    clear_btn.click(clear_chat, outputs=[chatbot, score_display])

if __name__ == "__main__":
    demo.queue().launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://18c0f24ccdfdadef63.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
